In [1]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Step 1: Create an imbalanced binary classification dataset
X, y = make_classification(n_samples=1000, n_features=10, n_informative=2, n_redundant=8, 
                           weights=[0.9, 0.1], flip_y=0, random_state=42)

np.unique(y, return_counts=True)

(array([0, 1]), array([900, 100]))

In [3]:
# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)

Handle class imbalance

In [4]:
from imblearn.combine import SMOTETomek

smt = SMOTETomek(random_state=42)
X_train_res, y_train_res = smt.fit_resample(X_train, y_train)
np.unique(y_train_res, return_counts=True)

(array([0, 1]), array([619, 619]))

In [5]:
models = [
    (
        "Logistic Regression", 
        {"C": 1, "solver": 'lbfgs'},
        LogisticRegression(), 
        (X_train, y_train),
        (X_test, y_test)
    ),
    (
        "Random Forest", 
        {"n_estimators": 30, "max_depth": 3},
        RandomForestClassifier(), 
        (X_train, y_train),
        (X_test, y_test)
    ),
    (
        "XGBClassifier",
        {"use_label_encoder": False, "eval_metric": 'logloss'},
        XGBClassifier(), 
        (X_train, y_train),
        (X_test, y_test)
    ),
    (
        "XGBClassifier With SMOTE",
        {"use_label_encoder": False, "eval_metric": 'logloss'},
        XGBClassifier(), 
        (X_train_res, y_train_res),
        (X_test, y_test)
    )
]

In [6]:
reports = []

for model_name, params, model, train_set, test_set in models:
    X_train = train_set[0]
    y_train = train_set[1]
    X_test = test_set[0]
    y_test = test_set[1]
    
    model.set_params(**params)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    report = classification_report(y_test, y_pred, output_dict=True)
    reports.append(report)

In [7]:
import mlflow
import mlflow.sklearn
import mlflow.xgboost

In [10]:
# Initialize MLflow

mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("deployment_experiment1")

for i, element in enumerate(models):
    model_name = element[0]
    params = element[1]
    model = element[2]
    report = reports[i]
    
    with mlflow.start_run(run_name=model_name):        
        mlflow.log_params(params)
        mlflow.log_metrics({
            'accuracy': report['accuracy'],
            'recall_class_1': report['1']['recall'],
            'recall_class_0': report['0']['recall'],
            'f1_score_macro': report['macro avg']['f1-score']
        })  
        
        if "XGB" in model_name:
            mlflow.xgboost.log_model(model, "model")
        else:
            mlflow.sklearn.log_model(model, "model")  

2026/08/12 01:35:34 INFO mlflow.tracking.fluent: Experiment with name 'deployment_experiment1' does not exist. Creating a new experiment.
2026/08/12 01:35:34 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/08/12 01:35:40 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run Logistic Regression at: http://127.0.0.1:5000/#/experiments/4/runs/6baba8415f7b4231868bb6e813a9a26d
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4


2026/08/12 01:35:44 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run Random Forest at: http://127.0.0.1:5000/#/experiments/4/runs/6f70f29bbb8242ad83c708c86ba3d7ff
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4


2026/08/12 01:35:46 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run XGBClassifier at: http://127.0.0.1:5000/#/experiments/4/runs/12827a343c88475f9930c6baaf6cc195
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4
🏃 View run XGBClassifier With SMOTE at: http://127.0.0.1:5000/#/experiments/4/runs/228214cc12b04af8a0d5e07b6832c64a
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4


Register the Model

In [11]:
model_name = 'XGB-Classifier'
run_id=input('Please type RunID')
model_uri = f'runs:/{run_id}/model'

with mlflow.start_run(run_id=run_id):
    mlflow.register_model(model_uri=model_uri, name=model_name)


Registered model 'XGB-Classifier' already exists. Creating a new version of this model...
2026/08/12 01:36:23 WARNING mlflow.tracking._model_registry.fluent: Run with id 12827a343c88475f9930c6baaf6cc195 has no artifacts at artifact path 'model', registering model based on models:/m-e0fd5c94f06047cea1b6f5e8dcce12bc instead
2026/08/12 01:36:23 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: XGB-Classifier, version 1


🏃 View run XGBClassifier at: http://127.0.0.1:5000/#/experiments/4/runs/12827a343c88475f9930c6baaf6cc195
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4


Created version '1' of model 'XGB-Classifier'.


Load the Model

In [12]:
model_name = 'XGB-Classifier'
model_version = 1
model_uri = f"models:/{model_name}/{model_version}"

loaded_model = mlflow.xgboost.load_model(model_uri)
y_pred = loaded_model.predict(X_test)
y_pred[:4]

array([0, 0, 0, 0])

Transition the Model to Production server

In [13]:
current_model_uri = f"models:/{model_name}@appserver"
production_model_name = "finalproduction"

client = mlflow.MlflowClient()
client.copy_model_version(src_model_uri=current_model_uri, dst_name=production_model_name)

MlflowException: Failed to fetch model version from source model URI: 'models:/XGB-Classifier@appserver'. Error: INVALID_PARAMETER_VALUE: Registered model alias appserver not found.

In [ ]:
'''
# ignore below code

model_version = 2
prod_model_uri = f"models:/{production_model_name}@productionserver2"

loaded_model = mlflow.xgboost.load_model(prod_model_uri)
y_pred = loaded_model.predict(X_test)
y_pred[:4]
'''